# Offline Study Assistant — Development Notebook
### SDG 4: Quality Education

**Problem:** Students in areas with unreliable or expensive internet access can't
rely on cloud-based AI tutors. This project builds an AI study assistant that runs
entirely offline (local model via Ollama), lets a student upload their own notes,
ask grounded questions about them (RAG), get quizzed on the material, and receive
suggestions on what to review next based on quiz performance (agentic layer).

**Approach:** RAG for grounded Q&A over the student's own notes, plus an agentic
layer (3 tools: quiz generation, weak-topic tracking, next-topic suggestion) that
the assistant invokes based on what the student is asking for.

This notebook documents data preparation, chunking/retrieval experiments, prompt
engineering iterations, model evaluation, and the agent implementation. The actual
production code lives in `../backend/app/` and is imported directly here so the
notebook always reflects the real pipeline.

## 1. Setup

In [ ]:
import sys
sys.path.append('../backend')

from app.rag import StudyIndex, chunk_text, build_prompt, call_ollama, answer_question
from app import agent, db

## 2. Data preparation

For development, we use a small set of sample study notes. Replace this cell with
your own real notes (biology, history, whatever subject you're demoing) before
your final submission.

In [ ]:
sample_documents = [
    {
        "text": (
            "Photosynthesis is the process by which green plants use sunlight to "
            "synthesize food from carbon dioxide and water. It occurs mainly in the "
            "chloroplasts, using chlorophyll to capture light energy. The process "
            "produces glucose and releases oxygen as a byproduct. It takes place in "
            "two stages: the light-dependent reactions and the light-independent "
            "reactions (Calvin cycle)."
        ),
        "source": "biology_notes.txt",
        "topic": "photosynthesis",
    },
    {
        "text": (
            "Cell division allows organisms to grow and repair damaged tissue. "
            "Mitosis produces two genetically identical daughter cells and is used "
            "for growth and repair. Meiosis produces four genetically distinct "
            "gametes and is used in sexual reproduction, introducing genetic "
            "variation through crossing over."
        ),
        "source": "biology_notes.txt",
        "topic": "cell_division",
    },
]

for doc in sample_documents:
    print(doc["topic"], "-", len(doc["text"].split()), "words")

## 3. Chunking experiments

We compare two chunk sizes to see how they affect the number of chunks and
retrieval granularity. Smaller chunks retrieve more precisely but may lose
surrounding context; larger chunks preserve context but can dilute relevance.

In [ ]:
for size in [100, 400]:
    chunks = chunk_text(sample_documents[0]["text"], chunk_size=size, overlap=20)
    print(f"chunk_size={size}: {len(chunks)} chunk(s)")
    for c in chunks:
        print("  -", c[:80], "...")

**Observation:** at `chunk_size=100`, the photosynthesis note (short, ~70 words)
stays as a single chunk anyway since it's shorter than the window. For longer real
notes, smaller chunks will produce multiple chunks per topic, letting retrieval
pull just the most relevant paragraph rather than an entire document. We settled
on `chunk_size=400, overlap=50` as the production default — large enough to keep
a full idea together for short study notes, small enough to stay relevant for
longer documents.

## 4. Building the index and testing retrieval

**Note:** this cell requires downloading the `all-MiniLM-L6-v2` embedding model
from Hugging Face on first run, so it needs an internet connection once (the
model is then cached locally and the assistant runs fully offline afterward).

In [ ]:
index = StudyIndex()
index.build(sample_documents)

test_queries = [
    "What does photosynthesis produce?",
    "What is the difference between mitosis and meiosis?",
]

for q in test_queries:
    results = index.search(q, top_k=2)
    print("Query:", q)
    for r in results:
        print(f"  score={r['score']:.3f} topic={r['topic']}  {r['text'][:70]}...")
    print()

## 5. Prompt engineering

**Naive first attempt** — just concatenate context and question with no
instructions:

In [ ]:
naive_prompt_template = "{context}\n\nQ: {question}\nA:"
print(naive_prompt_template.format(context="[retrieved chunks]", question="[question]"))

This naive version tended (in manual testing) to let the model answer from its
own general knowledge rather than the provided notes, and it didn't handle the
"not enough information" case at all.

**Refined prompt** (used in production, see `rag.py::build_prompt`) explicitly:
- Tells the model to use ONLY the given context
- Tells it to admit uncertainty rather than guess
- Labels each chunk with its source document

This reduced hallucination in manual testing and made answers traceable back to
a specific source file.

In [ ]:
print(build_prompt("What does photosynthesis produce?", index.search("What does photosynthesis produce?", top_k=1)))

## 6. Model evaluation

Compare 1-2 local models on the same set of test questions for speed and
groundedness. **Run this section on your own machine with Ollama running** —
it won't execute in this development environment.

Suggested models to compare, given typical laptop hardware (16GB RAM, no
dedicated GPU): `llama3.2:3b` and `phi3:mini`. Avoid 14B+ models per the
assignment's own guidance on inference performance.

In [ ]:
import time

models_to_compare = ["llama3.2:3b", "phi3:mini"]  # adjust to what you've pulled

for model in models_to_compare:
    print(f"--- {model} ---")
    for q in test_queries:
        start = time.time()
        result = answer_question(index, q)
        elapsed = time.time() - start
        print(f"Q: {q}")
        print(f"A: {result['answer']}")
        print(f"({elapsed:.1f}s)\n")

**Fill in after running:** which model was faster? Which stayed better grounded
in the provided notes? Which would you ship for the demo, and why?

## 7. Agentic layer

Three tools sit on top of the RAG core:

1. `agent.generate_quiz(index, topic, num_questions)` — generates quiz questions
   grounded in the notes for a given topic
2. `agent.track_weak_topics(student_id, topic, correct)` — logs a quiz attempt to SQLite
3. `agent.suggest_next_topic(student_id)` — recommends what to review based on
   accuracy history

A simple keyword router (`agent.route_message`) decides whether an incoming
message should trigger the quiz flow, the progress-suggestion flow, or a plain
RAG question. We chose manual routing over native function-calling because it's
transparent, easy to debug, and works reliably even with small local models that
may not support structured tool-calling well. The tradeoff is that it only
recognizes the keyword patterns we've defined — a broader intent classifier
would generalize better but adds complexity.

**Multi-model routing:** beyond the quiz/progress router, `answer_question()`
also checks whether a question looks mathematical (keywords like "solve",
"calculate", or a digit-operator pattern) and routes it to `deepseek-r1:1.5b`
instead of the general `llama3.2:latest` model, since reasoning-tuned models
tend to walk through steps more reliably on math. This is a second, independent
router operating on model choice rather than tool choice — worth calling out
as a design decision in its own right.

**Image ingestion:** notes can also be photographs of handwritten or printed
pages. `load_image_file()` sends the image to the local vision model
(`moondream:1.8b`) with a transcription prompt, and the returned text is
chunked/embedded exactly like any other document. This is a general vision
model rather than a dedicated OCR engine, so accuracy on dense or messy
handwriting will be lower than on clear print — worth testing with your own
photographed notes and noting the failure cases.

In [ ]:
db.init_db()

# Test routing
for msg in ["Can you quiz me on cell division?", "What should I study next?", "What is mitosis?"]:
    print(msg, "->", agent.route_message(msg))

In [ ]:
# Simulate a few quiz attempts and see the suggestion adapt
agent.track_weak_topics("demo_student", "cell_division", True)
agent.track_weak_topics("demo_student", "photosynthesis", False)
agent.track_weak_topics("demo_student", "photosynthesis", False)

print(agent.suggest_next_topic("demo_student"))

**Note:** `generate_quiz` requires Ollama running locally (it calls the model to
write questions grounded in the topic's notes). Run this on your own machine:

```python
questions = agent.generate_quiz(index, "photosynthesis", num_questions=3)
print(questions)
```

## 8. Exploratory work / limitations

- Chunking is word-count based rather than true token-based — good enough for
  this scope, but a tokenizer-aware chunker would be more precise for longer documents.
- The agent router uses fixed keyword lists; it will miss rephrased intents
  (e.g. "quiz" instead of "quiz me"). Expanding trigger phrases or using the
  model's own intent classification would generalize better.
- `generate_quiz`'s JSON parsing has a fallback for malformed model output, but
  small local models occasionally don't follow the JSON format instruction
  perfectly — worth tightening with a stricter prompt or a retry loop.
- Weak-topic detection is a simple accuracy threshold (<60%); a spaced-repetition
  style priority score would be a natural next improvement.
- Only single-student, single-device usage was tested — no concurrency testing.

## 9. Conclusion

This project shows that a genuinely useful, offline-capable AI study assistant is
achievable with lightweight local models, without depending on cloud APIs or
constant connectivity — directly relevant for students in low-connectivity areas
working toward SDG 4 (Quality Education). With more time, priorities would be:
a smarter agent router, spaced-repetition-style topic suggestions, and OCR
support so students can photograph handwritten notes instead of typing them.